# Chapter 1
## Modeling a Single Neuron
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter01.ipynb)

## About this chapter

This is the repository's first executable modeling example: a
conductance-based Hodgkin-Huxley (HH) simulation and voltage trace.

A membrane stores charge, so its voltage changes only when the injected and
ionic currents do not balance. Sodium activation provides rapid positive
feedback; sodium inactivation and potassium activation provide delayed
negative feedback. The three gating variables make those conductances depend
on both voltage and recent history.

The voltage balance implemented here is

$$
C\frac{dV}{dt}=I_{\mathrm{ext}}-g_{\mathrm{Na}}m^3h(V-E_{\mathrm{Na}})
-g_{\mathrm{K}}n^4(V-E_{\mathrm{K}})-g_{\mathrm{L}}(V-E_{\mathrm{L}}).
$$

Here $V$ is membrane voltage, $t$ is time, $C$ is membrane capacitance,
$I_{\mathrm{ext}}$ is applied current, $g_{\mathrm{Na}}$, $g_{\mathrm{K}}$,
and $g_{\mathrm{L}}$ are maximal sodium, potassium, and leak conductances,
and $E_{\mathrm{Na}}$, $E_{\mathrm{K}}$, and $E_{\mathrm{L}}$ are their
reversal potentials. $m$, $h$, and $n$ are dimensionless sodium-activation,
sodium-inactivation, and potassium-activation gates. Each gate
$x\in\{m,h,n\}$ follows $dx/dt=\alpha_x(V)(1-x)-\beta_x(V)x$, where
$\alpha_x$ and $\beta_x$ are voltage-dependent opening and closing rates.

See [`README.md`](chapter01.md) for the full guide,
including suggested order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact

## Hodgkin-Huxley Voltage Trace

The classical squid-axon Hodgkin-Huxley model, driven by a constant external current.

In [ ]:
def beta_n(v):
    return 0.125 * exp(-(v + 70.0) / 80.0)


def beta_m(v):
    return 4.0 * exp(-(v + 70.0) / 18.0)


def beta_h(v):
    return 1. / (exp(-(v + 40.0) / 10.0) + 1.0)


def alpha_n(v):
    return 0.01 * (-60.0 - v) / (exp((-60.0 - v) / 10.0) - 1.0)


def alpha_m(v):
    if np.abs(v + 45.0) > 1.0e-8:
        return (v + 45.0) / 10.0 / (1.0 - exp(-(v + 45.0) / 10.0))
    else:
        return 1.0


def alpha_h(v):
    return 0.07 * exp(-(v + 70) / 20)


def h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))


def simulate_hh_voltage_trace(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                               v_k=-82.0, v_na=45.0, v_l=-59.0,
                               i_ext=7.0, t_final=200.0, dt=0.01):
    def derivative(x0, t):
        v, m, n, h = x0
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l)) / c
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dm, dn, dh]

    v0 = -70.0
    x0 = [v0, m_inf(v0), n_inf(v0), h_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0]


def plot_hh_voltage_trace(t, v):
    plt.figure(figsize=(7, 3))
    plt.plot(t, v, lw=2, c="k")
    plt.xlim(min(t), max(t))
    plt.ylim(-100, 50)
    plt.xlabel("time [ms]")
    plt.ylabel("v [mV]")
    plt.yticks(range(-100, 100, 50))
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_voltage_trace(*simulate_hh_voltage_trace())

In [ ]:
interact(lambda i_ext=7.0: plot_hh_voltage_trace(*simulate_hh_voltage_trace(i_ext=i_ext)),
         i_ext=(0.0, 20.0, 0.5));